In [0]:
from pyspark.sql import functions as F

# Use the SAME names as your ingestion notebook (verify in Catalog)
BRONZE_TABLE     = "realtime_fraud_detection.`01_bronze`.raw_transactions"
SILVER_TABLE     = "realtime_fraud_detection.`02_silver`.cleaned_transactions"
QUARANTINE_TABLE = "realtime_fraud_detection.`02_silver`.quarantine_transactions"

bronze_df = spark.read.table(BRONZE_TABLE)
silver_df = spark.read.table(SILVER_TABLE)

bronze_count = bronze_df.count()
silver_count = silver_df.count()

# Quarantine table only exists if at least one bad row was ever written
if spark.catalog.tableExists(QUARANTINE_TABLE):
    quarantine_df = spark.read.table(QUARANTINE_TABLE)
    quarantine_count = quarantine_df.count()
else:
    quarantine_df = None
    quarantine_count = 0

print("📊 --- Row Counts ---")
print(f"Bronze:      {bronze_count}")
print(f"Silver:      {silver_count}")
print(f"Quarantine:  {quarantine_count}")

# --- Quarantine audit ---
if quarantine_count > 0:
    print("\n⚠️ Quarantine breakdown by reason:")
    display(quarantine_df.groupBy("_reason").count().orderBy(F.desc("count")))
    print("🔍 Sample quarantined rows:")
    display(quarantine_df.limit(10))

# --- Silver quality checks in a single pass ---
checks = silver_df.agg(
    F.sum(F.when((F.col("Amount") < 0) | F.isnan("Amount"), 1).otherwise(0)).alias("invalid_amounts"),
    F.sum(F.when(~F.col("Class").isin(0, 1), 1).otherwise(0)).alias("invalid_classes"),
    F.sum(F.when(F.col("Time").isNull() | F.col("Amount").isNull() | F.col("Class").isNull(), 1)
           .otherwise(0)).alias("null_core_cols"),
).collect()[0]

# Duplicate check: set to your business key columns if Silver has extra metadata columns
DEDUP_COLS = silver_df.columns
duplicate_rows = silver_count - silver_df.dropDuplicates(DEDUP_COLS).count()

print("\n--- Silver Data Quality Results ---")
print(f"Invalid/NaN amounts:  {checks['invalid_amounts']}")
print(f"Invalid class labels: {checks['invalid_classes']}")
print(f"Null core columns:    {checks['null_core_cols']}")
print(f"Duplicate rows:       {duplicate_rows}")

# --- Assertions ---
assert checks["invalid_amounts"] == 0, "Negative/NaN amounts found in Silver!"
assert checks["invalid_classes"] == 0, "Invalid classes found in Silver!"
assert checks["null_core_cols"] == 0, "Nulls found in Silver core columns!"
assert duplicate_rows == 0, f"{duplicate_rows} duplicate rows found in Silver!"
assert bronze_count == silver_count + quarantine_count, (
    f"Reconciliation failed: Bronze {bronze_count} != Silver {silver_count} + Quarantine {quarantine_count}"
)

print("\n✅ All Silver data quality and reconciliation checks passed!")